## Building the transaction network

This notebook reshapes the transaction data into a network so that each account's position within the wider flow of money can be measured.

In the audited table, every row describes a single transaction in isolation. That view captures the properties of individual payments but cannot express how accounts relate to one another, who pays whom, who acts as a hub, who sits at the end of a chain. Those relationships are often where the signal in financial-crime detection lies, and they only become visible once the data is expressed as a network.

The network is made of two elements. Accounts are represented as nodes, and transactions as directed, weighted connections between them: the direction records which account sent funds to which, and the weight records the amount. From this structure, later stages of the pipeline derive features that summarise each account's role in the network.

The network is built exclusively from transactions in the training period. Connections from the validation and test periods are deliberately excluded, so that no information from those periods can influence an account's representation. This preserves the strict separation established by the temporal split and ensures that the features derived here remain free of look-ahead information.

The notebook concludes by saving the constructed network and a reference table mapping each account to its position within it, ready for feature extraction in the next stage.

In [1]:
import os, sys, json, random, time
import numpy as np
import pandas as pd
import igraph as ig
from pathlib import Path

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path("..").resolve()
DATA_PROC = ROOT / "data/processed"
OUT_DIR = ROOT / "outputs"

print("igraph version:", ig.__version__)
print("Audit file present:", (DATA_PROC / "01_audit.parquet").exists())

igraph version: 0.11.9
Audit file present: True


In [2]:
# Load the audited transactions and ensure the timestamp is a datetime type.
df = pd.read_parquet(DATA_PROC / "01_audit.parquet")
df["Timestamp"] = pd.to_datetime(df["Timestamp"])
print(f"Loaded {len(df):,} transactions")

# Reconstruct the temporal split from timestamp percentiles (70/15/15).
# Recomputing from the timestamp guarantees each label aligns with its own transaction.
t70 = df["Timestamp"].quantile(0.70)
t85 = df["Timestamp"].quantile(0.85)
df["split"] = np.where(df["Timestamp"] <= t70, "train",
              np.where(df["Timestamp"] <= t85, "val", "test"))

# Verify the partition reproduces the one produced by the splitting notebook.
counts = df["split"].value_counts().to_dict()
assert counts == {"train": 3554957, "val": 761749, "test": 761639}, f"Split mismatch: {counts}"
print("Split reproduced:", counts)

Loaded 5,078,345 transactions
Split reproduced: {'train': 3554957, 'val': 761749, 'test': 761639}


In [3]:
# Cast account identifiers to strings so igraph treats them as vertex names, not vertex indices.
SENDER_COL, RECEIVER_COL = "Account", "Account.1"
df[SENDER_COL] = df[SENDER_COL].astype(str)
df[RECEIVER_COL] = df[RECEIVER_COL].astype(str)

# Restrict edges to training-period transactions to prevent look-ahead leakage in graph features.
df_train_edges = df[df["split"] == "train"].copy()
print(f"Training-period edges: {len(df_train_edges):,}")

# Construct a directed, weighted graph: accounts are vertices, transactions are edges,
# and the received amount is the edge weight.
edges = list(zip(df_train_edges[SENDER_COL], df_train_edges[RECEIVER_COL]))
weights = df_train_edges["Amount Received"].astype(float).tolist()

t0 = time.time()
g = ig.Graph.TupleList(edges, directed=True)
g.es["weight"] = weights
print(f"Build time: {time.time() - t0:.1f}s")
print(f"Vertices (accounts): {g.vcount():,}  Edges (transactions): {g.ecount():,}")

Training-period edges: 3,554,957
Build time: 3.9s
Vertices (accounts): 513,284  Edges (transactions): 3,554,957


In [4]:
# Makes sure the folder exists before writing to it
DATA_PROC.mkdir(parents=True, exist_ok=True)

# Saves the graph (accounts, 3.5M edges, weights) as one file
g.write_pickle(str(DATA_PROC / "04_graph_train.pkl"))

# Saves the account lookup table: igraph counts vertices 0,1,2...
# but your data uses account IDs, so this maps between them
account_map = pd.DataFrame({
    "account_id": g.vs["name"],
    "vertex_index": range(g.vcount()),
})
account_map.to_parquet(DATA_PROC / "04_account_map.parquet", index=False)

# Confirms both files are really on disk
for f in ["04_graph_train.pkl", "04_account_map.parquet"]:
    p = DATA_PROC / f
    assert p.exists(), f"MISSING: {f} - save failed, do not proceed"
    print(f"Ok {f}: {p.stat().st_size / 1e6:.1f} MB")

print(f"\nAccounts in map: {len(account_map):,}")

# Account IDs must stay text, not numbers, or the Day 5 join silently fails
print(f"Sample IDs: {account_map['account_id'].head(3).tolist()}")
print(f"Type: {account_map['account_id'].dtype}  (expect: object)")

Ok 04_graph_train.pkl: 77.9 MB
Ok 04_account_map.parquet: 5.8 MB

Accounts in map: 513,284
Sample IDs: ['8000EBD30', '8000F4580', '8000F5340']
Type: object  (expect: object)
